In [1]:
from typing import Callable

import pandas as pd
import polars as pl
from pathlib import Path
from datetime import datetime, date


def getBatch(inputFunction : Callable) -> Callable:
    def innerWrap(inSeries : pl.Series):
        return pl.Series(map(inputFunction, inSeries))
    return innerWrap

@getBatch
def stripString(x: str | None) -> str | None:
    if x:
        return x.strip()
    return None


loc_cols = ['GFPipeID', 'PipeName', 'GFLocID', 'GF_LocID_Tag', 'LocID', 'Loc', 'GF_LocName', 'LocName', 'GF_FacilityType', 'GF_FacilityTypeGroup', 'ReportedFacilityType', 'LocSegment', 'LocZone', 'State', 'County', 'InterconnectingEntity', 'UpdatedTime']

fact_cols =['GasDay','GFLocID','LocName','DesignCapacity','OperatingCapacity','TotalScheduledQuantity','OperationallyAvailableCapacity','IT','Direction','Timestamp']


Meta_path = Path('.').absolute().parent / 'downloads/enbridge/MetaData'

pipe_configs_path = Path('.').absolute().parent / 'src/artifacts/configFiles/PipeConfigs.parquet'

pipesDf = pl.read_parquet(pipe_configs_path)


In [2]:
temp_list =[]
for i in Meta_path.iterdir():
    pipe_code = i.name.split('_',1)[0].strip()
    temp = pd.read_csv(i,dtype=str)
    if not temp.empty:
        temp_list.append(temp.assign(PipeCode=pipe_code))
    
df_OA : pd.DataFrame= pd.concat(temp_list,ignore_index=True)
df = pl.from_dataframe(df=df_OA.astype(str))
df= df.with_columns(*[pl.col(col_name).str.strip_chars() for col_name in df.columns])


In [3]:
meta_loc_columns =[
 'Loc',
 'Loc Name',
 'Loc St Abbrev',
 'Loc Cnty',
 'Loc Zone',
 'Dir Flo',
 'Loc Type Ind',
  'PipeCode'
]

In [4]:
meta_locsDf = df.select(meta_loc_columns)\
    .join(pipesDf["PipeCode","GFPipeID","PipeName"], on="PipeCode").unique()\
    .with_columns(
        pl.concat_str(
            [
                pl.col("GFPipeID"),
                pl.col("Loc").str.strip_chars().str.zfill(7).cast(pl.String)
                
            ]
        ).alias("GFLocID")
    ).unique()


In [5]:
meta_locsDf.write_csv(pipe_configs_path.parent / "metaLocs.csv")